# مدرك: أين تقل الجمعيات الأهلية مقارنة بالسكان؟
**هاكثون تحليل البيانات في القطاع غير الربحي 2** · إعداد: حصة التركي

هذا الدفتر يشغّل وكيل «مدرك» كاملاً من الملفات الخام المفتوحة إلى التقرير:
تنظيف موثّق ← **توقف لموافقة الإنسان** ← تحليل الجمعيات لكل 100 ألف نسمة ← مصفوفة قرار ← نموذج تنبؤي لفجوة 2030 ← تقرير تفاعلي.

**طريقة التشغيل:** من القائمة `Runtime` ← `Run all`. التشغيل كاملاً يأخذ نحو 3 إلى 5 دقائق في Google Colab.

> مدرك يعدّ الجمعيات ولا يقيس أثرها. كل الأرقام مجمّعة على مستوى المنطقة والتخصص.

## ١. تحميل المشروع وتثبيت المكتبات

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if not os.path.exists('scripts/run_pipeline.py'):          # في Colab: نسحب المشروع من GitHub
    if not os.path.exists('mudrik'):
        !git clone -q https://github.com/hessahalturkiworks/mudrik.git
    %cd mudrik
!pip install -q -r requirements.txt
if IN_COLAB:   # متصفح لتحويل الرسوم والتقرير إلى PNG وPDF بعربية صحيحة، وخطوط عربية
    !playwright install --with-deps chromium > /dev/null 2>&1 && echo 'Chromium: جاهز'
    !apt-get -qq install -y fonts-noto-core > /dev/null 2>&1 && echo 'الخطوط العربية: جاهزة'
print('المجلد الحالي:', os.getcwd())

## ٢. البيانات الخام
كل الملفات سعودية مفتوحة، من المنصة الوطنية للبيانات المفتوحة والهيئة العامة للإحصاء. **مدرك لا يعدّل هذه الملفات أبداً.**

In [ ]:
import hashlib, pathlib
def md5(p): return hashlib.md5(pathlib.Path(p).read_bytes()).hexdigest()[:10]
RAW_BEFORE = {p.name: md5(p) for p in sorted(pathlib.Path('data_raw').iterdir()) if p.is_file()}
for name, h in RAW_BEFORE.items():
    print(f'{name:<48} {pathlib.Path("data_raw", name).stat().st_size/1024:>8.0f} KB   md5:{h}')

## ٣. التنظيف ثم التوقف لموافقة الإنسان
يُنظّف الوكيل ملفات الجمعيات والمؤسسات والسكان والمسح، ويسجّل كل تعديل، ولا يخمّن القيم المجهولة.
ثم **يتوقف** ويعرض القرارات (مثل استبعاد الجمعيات الوطنية وغير المصنفة)، ولا يحسب شيئاً قبل الموافقة.
رسالة `STOPPED for human review` في النهاية مقصودة.

In [ ]:
!python scripts/run_pipeline.py

## ٤. موافقة الإنسان وإكمال التحليل
بعد مراجعة القرارات أعلاه، نعيد التشغيل مع `--yes` (موافقة صاحبة المشروع)، فيكمل الوكيل:
الجمعيات لكل 100 ألف نسمة، ومصفوفة القرار، وعدسة الطلب، والنموذج التنبؤي لعام 2030، والتقرير.

In [ ]:
!python scripts/run_pipeline.py --skip-clean --yes
!python scripts/build_summary.py --out outputs

## ٥. النتائج الرئيسية

In [ ]:
import pandas as pd
T = 'outputs/tables/'
reg = pd.read_csv(T + 'region_summary.csv', encoding='utf-8-sig')
print('الجمعيات لكل 100 ألف نسمة حسب المنطقة (من الأقل):')
display(reg.sort_values('per_100k_all_groups').reset_index(drop=True))

In [ ]:
M = pd.read_csv(T + 'decision_matrix.csv', encoding='utf-8-sig')
print('مصفوفة القرار (130 خانة = 13 منطقة × 10 تخصصات):')
display(M.recommendation.value_counts().rename('عدد الخانات').to_frame())
print('خانات «أولوية دعم» بأرقام كافية، مرتبة:')
display(M[M.priority_rank.notna()].sort_values('priority_rank')[['region','group','associations','per_100k','national_per_100k','missing_to_reach_national','demand_rank']].reset_index(drop=True))

### النموذج التنبؤي: فجوة 2030
جرّبنا طريقتين للتوقع، واخترنا الأدق باختبار رجعي: درّبنا على 2010–2018، وتوقعنا 2022، ثم قارنّا بالتعداد الفعلي.

In [ ]:
display(pd.read_csv(T + 'forecast_log.csv', encoding='utf-8-sig'))
display(pd.read_csv(T + 'forecast_2030_regions.csv', encoding='utf-8-sig')[['region','associations','missing_today','missing_2030','pop_2022','pop_2030']].head(6))

## ٦. الرسوم

In [ ]:
from IPython.display import Image, display as show
for f in ['heatmap_per100k.png', 'decision_matrix.png']:
    p = 'outputs/figures/' + f
    if os.path.exists(p): show(Image(p, width=1100))
    else: print('غير متوفر (يحتاج Chromium):', f)

## ٧. التقرير التفاعلي
التقرير كاملاً داخل الدفتر، وفيه محاكي «ماذا لو؟» لتجربة إضافة جمعيات جديدة. وهو أيضاً في `outputs/report.html` ويعمل دون إنترنت.

In [ ]:
import html as _h
from IPython.display import HTML
HTML(f'<iframe srcdoc="{_h.escape(open("outputs/report.html", encoding="utf-8").read())}" style="width:100%;height:900px;border:1px solid #ddd"></iframe>')

## ٨. فحص إعادة التشغيل
نتأكد أن الأرقام هي نفسها المنشورة في التقرير، وأن الملفات الخام لم تتغير.

In [ ]:
tot = reg.associations.sum(); pop = reg.population_2022.sum()
checks = {
    'عدد الجمعيات في التحليل = 4,328': tot == 4328,
    'المتوسط الوطني = 13.45 لكل 100 ألف': round(tot / pop * 1e5, 2) == 13.45,
    'الشرقية = 6.87 لكل 100 ألف': round(float(reg.loc[reg.region_key == 'الشرقية', 'per_100k_all_groups'].iloc[0]), 2) == 6.87,
    'المصفوفة: 12 / 81 / 37': list(M.recommendation.value_counts().sort_values().values) == [12, 37, 81],
    'الملفات الخام لم تتغير': RAW_BEFORE == {p.name: md5(p) for p in sorted(pathlib.Path('data_raw').iterdir()) if p.is_file()},
}
for k, ok in checks.items(): print(('✅ ' if ok else '❌ ') + k)
print('\nكل الفحوص نجحت.' if all(checks.values()) else '\nفحص لم ينجح، راجعي outputs/pipeline_details.log')

## ٩. تنزيل المخرجات (اختياري)

In [ ]:
if IN_COLAB:
    !zip -rq mudrik_outputs.zip outputs data_clean
    from google.colab import files; files.download('mudrik_outputs.zip')
else:
    print('المخرجات في مجلد outputs/')